# RNA pocket detection and ligand docking with BioBB, fpocket and AutoDock Vina

**Reproducible, local-file workflow** adapted from the original Oxford RNA docking notebook (PDB **7LYJ**).

The notebook performs: **PDB preparation → fpocket pockets → pocket selection → docking box → ligand preparation → RNA receptor PDBQT → Vina → optional multi-ligand screening**.

> **No ZINC20/ZINC22 downloads.** All receptor PDBs and ligand SDFs are read from `data/`. This avoids ZINC CAPTCHA pages and makes the workflow suitable for a GitHub repository. See `README.md` for installation, folder layout, and data provenance. Computational docking scores are screening metrics, not measured binding affinities.

## 0. Prepare the project

1. Put your downloaded RNA PDB at `data/structures/7LYJ.pdb`.
2. Put your downloaded SDFs at `data/ligands/ZINC000003830390.sdf`, or keep the historical naming `data/ligands/tempZINC000003830390.sdf`.
3. Optionally put `Integrated.csv` under `data/` with a `Column1` header and one ligand ID per row.
4. Install the software using `conda env create -f environment.yml`, then `conda activate molecular-jupyter`, followed by `jupyter lab`.

**Google Colab:** Opening a notebook from GitHub does not automatically clone its input files or install the external binaries. In Colab, set `GITHUB_REPO_URL` below to your public repository so that the notebook clones its files. The native BioBB/`fpocket`/`obabel`/`vina` dependencies still need a compatible installation; local Conda is the tested workflow target. No downloads occur unless you explicitly provide a repository URL.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import shutil

# Leave empty for local JupyterLab. On Google Colab, set your public GitHub URL.
GITHUB_REPO_URL = ""    # Example: "https://github.com/USERNAME/rna-ligand-docking.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and GITHUB_REPO_URL:
    repository_name = GITHUB_REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
    clone_dir = Path("/content") / repository_name
    if not (clone_dir / "docking_project.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(clone_dir)], check=True)
    PROJECT_ROOT = clone_dir
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/content/rna-ligand-docking')]
    PROJECT_ROOT = next((p for p in candidates if (p / 'docking_project.py').is_file()), cwd)

if not (PROJECT_ROOT / 'docking_project.py').is_file():
    raise FileNotFoundError(
        'Project module not found. Start Jupyter in the repository, or set GITHUB_REPO_URL on Colab.'
    )
sys.path.insert(0, str(PROJECT_ROOT))
from docking_project import (find_sdf, discover_ligands, validate_sdf,
                             validate_pdbqt, inspect_pdbqt, best_affinity, run_batch, VINA_DEFAULTS)

DATA = PROJECT_ROOT / 'data'
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('In Colab:', IN_COLAB)

Repository: /work/RNA/rna-ligand-docking
In Colab: False


## 1. Set the input files and protocol parameters

Change only these variables to use another receptor, ligand, or pocket. `PRIMARY_LIGAND_ID` identifies a file already downloaded locally; it is **not** queried from ZINC. The pocket filter and box offset retain the original notebook's settings.

In [2]:
PDB_CODE = '7LYJ'
PRIMARY_LIGAND_ID = 'ZINC000003830390'
RAW_PDB = DATA / 'structures' / f'{PDB_CODE}.pdb'
LIGAND_DIR = DATA / 'ligands'
INTEGRATED_CSV = DATA / 'Integrated.csv'  # optional; header must be Column1

PH = 7.4
POCKET_VOLUME_RANGE = [400, 2000]
SELECTED_POCKET = 1  # change after reviewing the pocket summary
BOX_OFFSET = 12
VINA_PROPERTIES = dict(VINA_DEFAULTS)

print('Receptor:', RAW_PDB)
print('Primary SDF:', LIGAND_DIR / f'{PRIMARY_LIGAND_ID}.sdf')

Receptor: /work/RNA/rna-ligand-docking/data/structures/7LYJ.pdb
Primary SDF: /work/RNA/rna-ligand-docking/data/ligands/ZINC000003830390.sdf


In [3]:
required_bins = ('fpocket', 'obabel', 'vina', 'check_structure')
missing_bins = [name for name in required_bins if shutil.which(name) is None]
if missing_bins:
    raise RuntimeError('Missing executables on PATH: ' + ', '.join(missing_bins) +
                       '. Install the Conda environment from environment.yml.')
if not RAW_PDB.is_file():
    raise FileNotFoundError(f'Missing receptor: {RAW_PDB}. See data/structures/README.md')
primary_sdf = find_sdf(PRIMARY_LIGAND_ID, LIGAND_DIR)
validate_sdf(primary_sdf)  # also rejects a cached ZINC CAPTCHA page
print('Input files and required executables found.')

Input files and required executables found.


## 2. Prepare the RNA structure with PDBFixer

This follows the original notebook: remove heterogens (including waters/ions), avoid rebuilding absent terminal residues, fill missing atoms, and add hydrogens at nominal pH 7.4. **For RNA docking, inspect this structure carefully:** removing Mg²⁺ or other structural species can change the pocket, and RNA protonation/charge models need independent chemical validation.

In [4]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

processed_pdb = RESULTS / 'processed.pdb'
fixer = PDBFixer(filename=str(RAW_PDB))
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
chains = list(fixer.topology.chains())
for key in list(fixer.missingResidues):
    chain = chains[key[0]]
    if key[1] in (0, len(list(chain.residues()))):
        del fixer.missingResidues[key]
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(PH)
with processed_pdb.open('w') as handle:
    PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)
assert processed_pdb.stat().st_size > 0
print('Prepared RNA:', processed_pdb)

Prepared RNA: /work/RNA/rna-ligand-docking/results/processed.pdb


In [5]:
# Optional visualization: nglview frontend must work in your JupyterLab session.
import nglview as nv
view = nv.show_structure_file(str(processed_pdb))
view.clear_representations()
view.add_cartoon(color='lightblue')
view.center()
display(view)

/home/takeshi/miniconda3/envs/molecular-jupyter/lib/python3.11/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


NGLWidget()

## 3. Detect and visualize pockets with fpocket

The original notebook displayed the fpocket **cavity atom surfaces in red** and the **pocket PQR mesh in sky blue** (not the default ball-and-stick markers). This version restores those `add_surface()` representations. A blue **wireframe/contour** highlights each pocket; a semi-transparent red surface shows the cavity neighborhood. Available pocket IDs are printed below; use them to choose `SELECTED_POCKET` in Section 1.


In [6]:
import json
from biobb_vs.fpocket.fpocket_run import fpocket_run
from biobb_vs.fpocket.fpocket_filter import fpocket_filter

all_pockets = RESULTS / 'fpocket_all_pockets.zip'
pocket_summary = RESULTS / 'fpocket_summary.json'
filtered_pockets = RESULTS / 'fpocket_filtered.zip'
rc = fpocket_run(input_pdb_path=str(processed_pdb),
                 output_pockets_zip=str(all_pockets), output_summary=str(pocket_summary),
                 properties={'min_radius': 3, 'max_radius': 6, 'num_spheres': 35})
if rc != 0 or not pocket_summary.exists():
    raise RuntimeError(f'fpocket failed with exit code {rc}')
with pocket_summary.open() as f:
    print(json.dumps(json.load(f), indent=2)[:15000])
rc = fpocket_filter(input_pockets_zip=str(all_pockets),
                    input_summary=str(pocket_summary),
                    output_filter_pockets_zip=str(filtered_pockets),
                    properties={'volume': POCKET_VOLUME_RANGE})
if rc != 0 or not filtered_pockets.exists():
    raise RuntimeError('Pocket filtering failed or produced no output. Adjust POCKET_VOLUME_RANGE.')

2026-11-08 16:27:26,157 [MainThread  ] [INFO ]  Module: biobb_vs.fpocket.fpocket_run Version: 5.3.0
2026-11-08 16:27:26,158 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_ef933ade-9700-439b-b9c5-7768cbbd4088
2026-11-08 16:27:26,158 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/processed.pdb --> sandbox_ef933ade-9700-439b-b9c5-7768cbbd4088
2026-11-08 16:27:26,158 [MainThread  ] [INFO ]  Creating ac59e24e-5011-4a9f-a63d-f56f690c412d temporary folder
2026-11-08 16:27:26,159 [MainThread  ] [INFO ]  Executing fpocket
2026-11-08 16:27:26,159 [MainThread  ] [INFO ]  Launching command (it may take a while): cd ac59e24e-5011-4a9f-a63d-f56f690c412d ; fpocket -f input.pdb -m 3 -M 6 -i 35
2026-11-08 16:27:26,428 [MainThread  ] [INFO ]  Command 'cd ac59e24e-5011-4a9f-a63d-f56f690c412d ; fpocket -f input.pdb -m 3 -M 6 -i 35...' finalized with exit code 0
2026-11-08 16:27:26,429 [MainThread  ] [INFO ]  ***** POCKET 

In [7]:
# Restore the original pocket surface/mesh representations (not default dots).
import zipfile
import re

pockets_dir = RESULTS / 'pockets'
pockets_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(filtered_pockets) as archive:
    pocket_files = [name for name in archive.namelist()
                    if name.lower().endswith(('.pdb', '.pqr'))]
    if not pocket_files:
        raise RuntimeError('No pockets passed the volume filter')
    archive.extractall(pockets_dir)

pocket_ids = sorted({int(m.group(1)) for name in pocket_files
                     if (m := re.search(r'pocket(\d+)_', Path(name).name))})
print('Available filtered pocket IDs:', pocket_ids)
print('Configured SELECTED_POCKET:', SELECTED_POCKET)
if SELECTED_POCKET not in pocket_ids:
    print('WARNING: Selected pocket not present; change SELECTED_POCKET in Section 1.')

pocket_view = nv.NGLWidget()
protein_component = pocket_view.add_component(str(processed_pdb))
protein_component.clear_representations()
protein_component.add_cartoon(color='#cccccc', opacity=0.45)

num_cavity_surfaces = 0
num_pocket_meshes = 0
for pocket_file in sorted(pockets_dir.rglob('*')):
    if not pocket_file.is_file():
        continue
    if not re.search(r'pocket\d+_', pocket_file.name):
        continue
    if pocket_file.name.endswith('_atm.pdb'):
        cavity_component = pocket_view.add_component(str(pocket_file))
        cavity_component.clear_representations()
        cavity_component.add_surface(color='#cc0000', radius=1.5,
                                     lowResolution=True, smooth=1,
                                     opacity=0.3)
        num_cavity_surfaces += 1
    elif pocket_file.suffix.lower() == '.pqr':
        pocket_component = pocket_view.add_component(str(pocket_file))
        pocket_component.clear_representations()
        pocket_component.add_surface(color='skyblue', surfaceType='av',
                                     contour=True, smooth=1, opacity=0.65)
        num_pocket_meshes += 1

print(f'Cavity surfaces: {num_cavity_surfaces}; pocket meshes: {num_pocket_meshes}')
if num_pocket_meshes == 0:
    print('Warning: No .pqr files found. Inspect the filtered fpocket archive.')
pocket_view.center()
pocket_view._set_size('100%', '700px')
display(pocket_view)


Available filtered pocket IDs: [1, 2, 3]
Configured SELECTED_POCKET: 1
Cavity surfaces: 3; pocket meshes: 3


NGLWidget()

## 4. Select one pocket and generate the docking box

Set `SELECTED_POCKET` above, then rerun this cell. The `box.pdb` file is passed directly to BioBB's Vina wrapper, which computes the center and dimensions. This preserves the original approach instead of hard-coding one receptor's box coordinates.

In [8]:
from biobb_vs.fpocket.fpocket_select import fpocket_select
from biobb_vs.utils.box import box

pocket_cavity = RESULTS / 'fpocket_cavity.pdb'
pocket_points = RESULTS / 'fpocket_pocket.pqr'
docking_box = RESULTS / 'box.pdb'
if SELECTED_POCKET not in pocket_ids:
    raise ValueError(f'Choose a pocket from {pocket_ids}')
rc = fpocket_select(input_pockets_zip=str(filtered_pockets),
                   output_pocket_pdb=str(pocket_cavity),
                   output_pocket_pqr=str(pocket_points),
                   properties={'pocket': int(SELECTED_POCKET)})
if rc != 0 or not pocket_points.exists():
    raise RuntimeError('fpocket_select failed')
rc = box(input_pdb_path=str(pocket_points), output_pdb_path=str(docking_box),
         properties={'offset': BOX_OFFSET, 'box_coordinates': True})
if rc != 0 or not docking_box.exists():
    raise RuntimeError('Docking-box generation failed')
print('Docking box:', docking_box)

2026-11-08 16:27:48,599 [MainThread  ] [INFO ]  Module: biobb_vs.fpocket.fpocket_select Version: 5.3.0
2026-11-08 16:27:48,599 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_76bcaf8c-9c61-42f6-9a27-a38dfcf8f4d0
2026-11-08 16:27:48,600 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/fpocket_filtered.zip --> sandbox_76bcaf8c-9c61-42f6-9a27-a38dfcf8f4d0
2026-11-08 16:27:48,600 [MainThread  ] [INFO ]  Creating fa15ce1f-3431-4e11-ae37-80709c26fa70 temporary folder
2026-11-08 16:27:48,601 [MainThread  ] [INFO ]  Extracting: /work/RNA/rna-ligand-docking/results/fpocket_filtered.zip
2026-11-08 16:27:48,601 [MainThread  ] [INFO ]  to:
2026-11-08 16:27:48,601 [MainThread  ] [INFO ]  ['fa15ce1f-3431-4e11-ae37-80709c26fa70/pocket1_atm.pdb', 'fa15ce1f-3431-4e11-ae37-80709c26fa70/pocket1_vert.pqr', 'fa15ce1f-3431-4e11-ae37-80709c26fa70/pocket2_atm.pdb', 'fa15ce1f-3431-4e11-ae37-80709c26fa70/pocket2_vert.pqr', 'fa15ce

In [9]:
# Inspect the selected pocket as a surface, together with the Vina docking box.
box_view = nv.NGLWidget()
protein_component = box_view.add_component(str(processed_pdb))
protein_component.clear_representations()
protein_component.add_cartoon(color='#cccccc', opacity=0.35)

selected_pocket_component = box_view.add_component(str(pocket_points))
selected_pocket_component.clear_representations()
selected_pocket_component.add_surface(color='skyblue', surfaceType='av',
                                      lowResolution=True, smooth=1,
                                      contour=True, opacity=0.6)

box_component = box_view.add_component(str(docking_box))
box_component.clear_representations()
# Preserve the 12 connecting edges in the original notebook's docking box.
box_atom_pairs = [
    ['9999:Z.ZN1', '9999:Z.ZN2'], ['9999:Z.ZN2', '9999:Z.ZN4'],
    ['9999:Z.ZN4', '9999:Z.ZN3'], ['9999:Z.ZN3', '9999:Z.ZN1'],
    ['9999:Z.ZN5', '9999:Z.ZN6'], ['9999:Z.ZN6', '9999:Z.ZN8'],
    ['9999:Z.ZN8', '9999:Z.ZN7'], ['9999:Z.ZN7', '9999:Z.ZN5'],
    ['9999:Z.ZN1', '9999:Z.ZN5'], ['9999:Z.ZN2', '9999:Z.ZN6'],
    ['9999:Z.ZN3', '9999:Z.ZN7'], ['9999:Z.ZN4', '9999:Z.ZN8'],
]
box_component.add_representation('distance', atomPair=box_atom_pairs,
                                 labelVisible=False, color='black')
box_component.add_representation('ball+stick', selection='9999', color='pink',
                                 aspectRatio=8)
box_view.center()
box_view._set_size('100%', '700px')
display(box_view)


NGLWidget()

## 5. Prepare one ligand using existing SDF coordinates

Converts an existing **single-molecule** SDF to PDB and then PDBQT, using Open Babel through BioBB. We do not generate a new conformer or contact any external server. Check protonation, bond orders, stereochemistry, and charges before interpreting Vina results.

In [10]:
from biobb_chemistry.babelm.babel_convert import babel_convert

single_dir = RESULTS / 'single'
single_dir.mkdir(parents=True, exist_ok=True)
ligand_pdb = single_dir / 'ligand.pdb'
ligand_pdbqt = single_dir / 'prep_ligand.pdbqt'
for input_path, output_path, fmt_in, fmt_out in [
    (primary_sdf, ligand_pdb, 'sdf', 'pdb'),
    (ligand_pdb, ligand_pdbqt, 'pdb', 'pdbqt'),
]:
    rc = babel_convert(input_path=str(input_path), output_path=str(output_path),
                       properties={'input_format': fmt_in,
                                   'output_format': fmt_out, 'binary_path': 'obabel'})
    if rc != 0 or not output_path.exists():
        raise RuntimeError(f'Ligand conversion failed: {fmt_in} -> {fmt_out}')
print('Ligand PDBQT:', ligand_pdbqt)

2026-11-08 16:27:58,815 [MainThread  ] [INFO ]  Module: biobb_chemistry.babelm.babel_convert Version: 5.3.2
2026-11-08 16:27:58,815 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_09cbdce3-aea1-417e-9e1f-813684cc484e
2026-11-08 16:27:58,815 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/data/ligands/tempZINC000003830390.sdf --> sandbox_09cbdce3-aea1-417e-9e1f-813684cc484e
2026-11-08 16:27:58,815 [MainThread  ] [INFO ]  Value  is not compatible as a coordinates value
2026-11-08 16:27:58,816 [MainThread  ] [INFO ]  Launching command (it may take a while): obabel -isdf /work/RNA/rna-ligand-docking/notebooks/sandbox_09cbdce3-aea1-417e-9e1f-813684cc484e/tempZINC000003830390.sdf -opdb -O/work/RNA/rna-ligand-docking/notebooks/sandbox_09cbdce3-aea1-417e-9e1f-813684cc484e/ligand.pdb -r
2026-11-08 16:27:58,826 [MainThread  ] [INFO ]  Command 'obabel -isdf /work/RNA/rna-ligand-docking/notebooks/sandbox_09cbdce3-aea1-417e-

2026-11-08 16:27:58,827 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:27:58,828 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_09cbdce3-aea1-417e-9e1f-813684cc484e']
2026-11-08 16:27:58,828 [MainThread  ] [INFO ]  
2026-11-08 16:27:58,829 [MainThread  ] [INFO ]  Module: biobb_chemistry.babelm.babel_convert Version: 5.3.2
2026-11-08 16:27:58,829 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_0696641c-bc87-44d4-a9e0-d4e4128bfc07
2026-11-08 16:27:58,829 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/single/ligand.pdb --> sandbox_0696641c-bc87-44d4-a9e0-d4e4128bfc07
2026-11-08 16:27:58,830 [MainThread  ] [INFO ]  Value  is not compatible as a coordinates value
2026-11-08 16:27:58,830 [MainThread  ] [INFO ]  Launching command (it may take a while): obabel -ipdb /work/RNA/rna-ligand-docking/notebooks/sandbox_0696641c-bc87-44d4-a9e0-d4e4128bfc07/ligand.pdb -opdbqt -O/work/RNA/rna-ligand-docking/notebooks/sandbox_0696641c-bc87-44d

2026-11-08 16:27:58,839 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:27:58,840 [MainThread  ] [INFO ]  Removed 2 non PDBQT records (copied from the input by Open Babel) from /work/RNA/rna-ligand-docking/results/single/prep_ligand.pdbqt
2026-11-08 16:27:58,840 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_0696641c-bc87-44d4-a9e0-d4e4128bfc07']
2026-11-08 16:27:58,841 [MainThread  ] [INFO ]  
Ligand PDBQT: /work/RNA/rna-ligand-docking/results/single/prep_ligand.pdbqt


## 6. Prepare and validate the RNA receptor PDBQT

The original BioBB `check_structure` method is attempted first (`charges=True`, `mode='auto'`). On your machine it generated a file for which only **172 of 2051 atom records** passed the PDBQT fixed-column check, despite exit code 0. The reason might be incorrect formatting or missing PDBQT charges/types. **We cannot identify the exact cause without the actual output file.**

This cell now prints sample invalid records. If BioBB's output is malformed, it **preserves a copy** and tries an **explicit Open Babel rigid-receptor conversion** (`-xr -xn`) as a practical fallback. Open Babel is a *different charge/typing procedure* and can change the receptor electrostatics; it must not be assumed equivalent to BioBB for scientific comparisons. Review phosphate groups, protonation and atom types before interpreting RNA docking scores. The validator checks format only, not chemical accuracy.


In [11]:
from biobb_structure_utils.utils.str_check_add_hydrogens import str_check_add_hydrogens
import subprocess
import shutil

receptor_pdbqt = RESULTS / 'prep_receptor.pdbqt'
if receptor_pdbqt.exists():
    receptor_pdbqt.unlink()  # avoid reusing a previous run accidentally
biobb_receptor = RESULTS / 'prep_receptor_biobb.pdbqt'
rc = str_check_add_hydrogens(input_structure_path=str(processed_pdb),
                             output_structure_path=str(biobb_receptor),
                             properties={'charges': True, 'mode': 'auto'})

# A process exit code of zero does not guarantee correctly formatted PDBQT.
if rc == 0 and biobb_receptor.is_file():
    details = inspect_pdbqt(biobb_receptor)
    print('BioBB PDBQT record check:', {k: v for k, v in details.items() if k != 'bad_examples'})
    if details['bad_examples']:
        print('Examples of invalid BioBB records:')
        for example in details['bad_examples'][:3]:
            print(example)
    if details['bad_atom_count'] == 0:
        shutil.copy2(biobb_receptor, receptor_pdbqt)
        print('Receptor preparation method: BioBB check_structure')
    else:
        print('BioBB produced incomplete/malformed PDBQT fields. Trying Open Babel.')
else:
    print(f'BioBB did not create valid output (exit code {rc}). Trying Open Babel.')

if not receptor_pdbqt.is_file():
    # Open Babel `-xr` writes a rigid PDBQT receptor; `-xn` preserves atom names.
    cmd = ['obabel', '-ipdb', str(processed_pdb), '-opdbqt', '-O',
           str(receptor_pdbqt), '-xr', '-xn']
    result = subprocess.run(cmd, text=True, capture_output=True)
    print('Open Babel command:', ' '.join(cmd))
    print('Open Babel output:', result.stderr.strip() or result.stdout.strip())
    if result.returncode != 0:
        raise RuntimeError(f'Open Babel receptor conversion failed ({result.returncode})')
    print('Receptor preparation method: Open Babel fallback (different charge model)')

print('PDBQT format check:', validate_pdbqt(receptor_pdbqt))
print('Receptor PDBQT:', receptor_pdbqt)
print('IMPORTANT: Validate RNA phosphate charges and atom types before production docking.')


2026-11-08 16:28:06,367 [MainThread  ] [INFO ]  Module: biobb_structure_utils.utils.str_check_add_hydrogens Version: 5.3.0
2026-11-08 16:28:06,367 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_35ad878c-d44e-479c-906c-b71ff27ffc6f
2026-11-08 16:28:06,368 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/processed.pdb --> sandbox_35ad878c-d44e-479c-906c-b71ff27ffc6f
2026-11-08 16:28:06,368 [MainThread  ] [INFO ]  Launching command (it may take a while): check_structure -i /work/RNA/rna-ligand-docking/notebooks/sandbox_35ad878c-d44e-479c-906c-b71ff27ffc6f/processed.pdb -o /work/RNA/rna-ligand-docking/notebooks/sandbox_35ad878c-d44e-479c-906c-b71ff27ffc6f/prep_receptor_biobb.pdbqt --non_interactive --force_save command_list --list 'add_hydrogen --add_charges ADT --add_mode auto '
2026-11-08 16:28:06,554 [MainThread  ] [INFO ]  Command 'check_structure -i /work/RNA/rna-ligand-docking/notebooks/sandbox_35ad878

## 7. Run a single AutoDock Vina calculation

Defaults match the earlier BioBB notebook: 1 CPU, exhaustiveness 8, 9 modes, minimum RMSD 1, energy range 3. This workflow does **not** set a Vina random seed; scores/poses can vary between runs or Vina versions.

In [12]:
from biobb_vs.vina.autodock_vina_run import autodock_vina_run

single_poses = single_dir / 'output_vina.pdbqt'
single_log = single_dir / 'output_vina.log'
rc = autodock_vina_run(input_ligand_pdbqt_path=str(ligand_pdbqt),
                       input_receptor_pdbqt_path=str(receptor_pdbqt),
                       input_box_path=str(docking_box),
                       output_pdbqt_path=str(single_poses),
                       output_log_path=str(single_log),
                       properties=VINA_PROPERTIES)
if rc != 0 or not single_poses.exists():
    raise RuntimeError(f'Vina docking failed with exit code {rc}; check {single_log}')
print('Best Vina affinity (kcal/mol):', best_affinity(single_poses))
print('Pose file:', single_poses)

2026-11-08 16:28:15,038 [MainThread  ] [INFO ]  Module: biobb_vs.vina.autodock_vina_run Version: 5.3.0
2026-11-08 16:28:15,038 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_c18c5086-fdf9-4c9f-8071-a316af83a193
2026-11-08 16:28:15,039 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/single/prep_ligand.pdbqt --> sandbox_c18c5086-fdf9-4c9f-8071-a316af83a193
2026-11-08 16:28:15,039 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/prep_receptor.pdbqt --> sandbox_c18c5086-fdf9-4c9f-8071-a316af83a193
2026-11-08 16:28:15,039 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/box.pdb --> sandbox_c18c5086-fdf9-4c9f-8071-a316af83a193
2026-11-08 16:28:15,040 [MainThread  ] [INFO ]  Launching command (it may take a while): cd /work/RNA/rna-ligand-docking/notebooks/sandbox_c18c5086-fdf9-4c9f-8071-a316af83a193 ; vina --ligand prep_ligand.pdbqt --receptor prep_recepto

In [13]:
# Optional: view docked poses (notebook may be slow when all poses are shown).
pose_view = nv.show_structure_file(str(single_poses))
pose_view.add_ball_and_stick()
pose_view.center()
display(pose_view)

NGLWidget()

## 8. Batch virtual screening from *local* SDF files

If `data/Integrated.csv` exists, the `Column1` IDs determine which SDFs are used; otherwise the helper discovers all `*.sdf` in `data/ligands/`. Filenames `tempZINC...sdf` are supported for compatibility with the old code.

**Start with `MAX_LIGANDS=5`** to check the workflow, then set `None` for all ligands. The default `MAX_WORKERS=1` avoids the notebook's previous `multiprocessing.Pool` error; with the module-level worker you can try `3` once serial runs work. Errors are stored in the result CSV rather than being hidden by `MaybeEncodingError`. Reruns skip ligands with a valid existing docking score.

In [14]:
MAX_LIGANDS = 5  # set to None to screen every local ligand
MAX_WORKERS = 1  # increase to 3 after successful serial docking

ligand_ids = discover_ligands(LIGAND_DIR, INTEGRATED_CSV, csv_column='Column1')
if not ligand_ids:
    raise RuntimeError('No ligands found. Copy local SDF files into data/ligands/.')
if MAX_LIGANDS is not None:
    ligand_ids = ligand_ids[:MAX_LIGANDS]
print(f'About to screen {len(ligand_ids)} ligands:', ligand_ids[:10])

About to screen 5 ligands: ['ZINC000003831551', 'ZINC000000000353', 'ZINC000003815424', 'ZINC000003872994', 'ZINC000058581064']


In [15]:
batch_dir = RESULTS / 'batch'
records = run_batch(ligand_ids, ligand_dir=LIGAND_DIR,
                    receptor=receptor_pdbqt, box=docking_box,
                    out_dir=batch_dir, max_workers=MAX_WORKERS,
                    vina_props=VINA_PROPERTIES, skip_existing=True)
print('Saved results:', batch_dir / 'results.csv')
print('Successful/skipped:', sum(r['status'] in ('ok', 'skipped') for r in records))
print('Errors:', sum(r['status'] == 'error' for r in records))

2026-11-08 16:28:41,608 [MainThread  ] [INFO ]  Module: biobb_chemistry.babelm.babel_convert Version: 5.3.2
2026-11-08 16:28:41,609 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_e7acd5d7-209b-4a41-b803-114f93bbf9ad
2026-11-08 16:28:41,609 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/data/ligands/tempZINC000003831551.sdf --> sandbox_e7acd5d7-209b-4a41-b803-114f93bbf9ad
2026-11-08 16:28:41,609 [MainThread  ] [INFO ]  Value  is not compatible as a coordinates value
2026-11-08 16:28:41,609 [MainThread  ] [INFO ]  Launching command (it may take a while): obabel -isdf /work/RNA/rna-ligand-docking/notebooks/sandbox_e7acd5d7-209b-4a41-b803-114f93bbf9ad/tempZINC000003831551.sdf -opdb -O/work/RNA/rna-ligand-docking/notebooks/sandbox_e7acd5d7-209b-4a41-b803-114f93bbf9ad/ligand.pdb -r
2026-11-08 16:28:41,620 [MainThread  ] [INFO ]  Command 'obabel -isdf /work/RNA/rna-ligand-docking/notebooks/sandbox_e7acd5d7-209b-4a41-

2026-11-08 16:28:41,621 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:28:41,621 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_e7acd5d7-209b-4a41-b803-114f93bbf9ad']
2026-11-08 16:28:41,621 [MainThread  ] [INFO ]  
2026-11-08 16:28:41,622 [MainThread  ] [INFO ]  Module: biobb_chemistry.babelm.babel_convert Version: 5.3.2
2026-11-08 16:28:41,622 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_4059c40e-d0c9-43b2-9f68-3ea81b5eeec8
2026-11-08 16:28:41,623 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/batch/ZINC000003831551/ligand.pdb --> sandbox_4059c40e-d0c9-43b2-9f68-3ea81b5eeec8
2026-11-08 16:28:41,623 [MainThread  ] [INFO ]  Value  is not compatible as a coordinates value
2026-11-08 16:28:41,623 [MainThread  ] [INFO ]  Launching command (it may take a while): obabel -ipdb /work/RNA/rna-ligand-docking/notebooks/sandbox_4059c40e-d0c9-43b2-9f68-3ea81b5eeec8/ligand.pdb -opdbqt -O/work/RNA/rna-ligand-docking/notebooks/sandbox_4

2026-11-08 16:28:41,636 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:28:41,637 [MainThread  ] [INFO ]  Removed 2 non PDBQT records (copied from the input by Open Babel) from /work/RNA/rna-ligand-docking/results/batch/ZINC000003831551/ligand.pdbqt
2026-11-08 16:28:41,637 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_4059c40e-d0c9-43b2-9f68-3ea81b5eeec8']
2026-11-08 16:28:41,637 [MainThread  ] [INFO ]  
2026-11-08 16:28:41,638 [MainThread  ] [INFO ]  Module: biobb_vs.vina.autodock_vina_run Version: 5.3.0
2026-11-08 16:28:41,638 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_a01e8cf6-4cb6-44ce-a17a-e4bf4396b239
2026-11-08 16:28:41,639 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/batch/ZINC000003831551/ligand.pdbqt --> sandbox_a01e8cf6-4cb6-44ce-a17a-e4bf4396b239
2026-11-08 16:28:41,639 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/prep_receptor.pdbqt --> sandbox_a01e8cf6-4cb6-44ce-a17a-e4bf43

2026-11-08 16:28:44,515 [MainThread  ] [INFO ]  1 molecule converted



... (11 outputs left out)

2026-11-08 16:29:45,969 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:29:45,969 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_11384426-9285-40df-8733-beca04d34a26']
2026-11-08 16:29:45,969 [MainThread  ] [INFO ]  
2026-11-08 16:29:45,970 [MainThread  ] [INFO ]  Module: biobb_chemistry.babelm.babel_convert Version: 5.3.2
2026-11-08 16:29:45,970 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_ae0f02d9-257e-4b6d-a042-c16d4b8026b7
2026-11-08 16:29:45,970 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/batch/ZINC000058581064/ligand.pdb --> sandbox_ae0f02d9-257e-4b6d-a042-c16d4b8026b7
2026-11-08 16:29:45,970 [MainThread  ] [INFO ]  Value  is not compatible as a coordinates value
2026-11-08 16:29:45,970 [MainThread  ] [INFO ]  Launching command (it may take a while): obabel -ipdb /work/RNA/rna-ligand-docking/notebooks/sandbox_ae0f02d9-257e-4b6d-a042-c16d4b8026b7/ligand.pdb -opdbqt -O/work/RNA/rna-ligand-docking/notebooks/sandbox_a

2026-11-08 16:29:45,980 [MainThread  ] [INFO ]  1 molecule converted



2026-11-08 16:29:45,980 [MainThread  ] [INFO ]  Removed 2 non PDBQT records (copied from the input by Open Babel) from /work/RNA/rna-ligand-docking/results/batch/ZINC000058581064/ligand.pdbqt
2026-11-08 16:29:45,980 [MainThread  ] [INFO ]  Removed: ['/work/RNA/rna-ligand-docking/notebooks/sandbox_ae0f02d9-257e-4b6d-a042-c16d4b8026b7']
2026-11-08 16:29:45,980 [MainThread  ] [INFO ]  
2026-11-08 16:29:45,981 [MainThread  ] [INFO ]  Module: biobb_vs.vina.autodock_vina_run Version: 5.3.0
2026-11-08 16:29:45,981 [MainThread  ] [INFO ]  Directory successfully created: /work/RNA/rna-ligand-docking/notebooks/sandbox_80ca77b7-0e83-40f7-95f5-a9a255ff8ccb
2026-11-08 16:29:45,981 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/batch/ZINC000058581064/ligand.pdbqt --> sandbox_80ca77b7-0e83-40f7-95f5-a9a255ff8ccb
2026-11-08 16:29:45,981 [MainThread  ] [INFO ]  Copy to stage: /work/RNA/rna-ligand-docking/results/prep_receptor.pdbqt --> sandbox_80ca77b7-0e83-40f7-95f5-a9a255

In [ ]:
import pandas as pd
summary = pd.read_csv(batch_dir / 'results.csv')
display(summary.head(20))
if (summary['status'] == 'error').any():
    print('Failures (see individual vina.log files for command output):')
    display(summary.loc[summary['status'] == 'error', ['ligand_id', 'error']].head(20))

## 9. Reproducibility and limitations

- **Input coordinates:** use the original downloaded PDB and SDF files to reproduce prior work; substituting new conformers or protonation states changes the problem.
- **RNA chemistry:** this tutorial-like pipeline does not validate receptor charge assignment, RNA flexibility, metal-ion dependence, or the suitability of Vina's protein-oriented scoring for RNA.
- **Pocket choices:** pocket numbering and box location depend on the receptor coordinates and fpocket settings.
- **Versions:** record your installed executable versions (`vina --version`, `obabel -V`, `fpocket -h`) and Conda environment (`conda env export --from-history`).
- **Results:** generated files go to `results/` (excluded from Git), and `results/batch/results.csv` stores per-ligand status and best docking score.

For a runnable command-line input check, use `python scripts/validate_inputs.py` from the project root. GitHub Actions only tests local-file helper functions, not expensive docking or correctness of RNA force-field choices.